# 04 — Final Forecasting Model and Bundle
This notebook performs final training, freeze, one-time holdout evaluation, and Notebook 05 handoff. It does not perform model selection.

## 1. Finalization Context and One-Time Holdout Boundary

In [1]:
from pathlib import Path
import hashlib, json, sys
import numpy as np
import pandas as pd
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'scripts/project_context.py').is_file(): PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.project_context import get_project_context
from scripts.forecasting_model_selection import load_and_validate_forecasting_model_selection_handoff
from scripts.forecasting_finalization import (run_forecasting_finalization, load_and_validate_forecasting_final_model_handoff, load_and_validate_forecasting_inference_bundle, load_and_validate_forecasting_final_test_evidence, load_and_validate_forecasting_final_model_manifest, load_trusted_forecasting_model_from_bundle)
PROJECT = get_project_context(start=PROJECT_ROOT)
UPSTREAM = PROJECT.root / 'artifacts/model-selection/nottem/model-selection-handoff.json'
OUTPUT = PROJECT.root / 'artifacts/models/nottem'
print({'project_root': str(PROJECT.root), 'boundary': 'holdout targets only after trusted staging freeze'})

{'project_root': '/home/fabyuu/Projetos/DATASET-ANALISYS/dataset-study-nottingham-monthly-temperatures', 'boundary': 'holdout targets only after trusted staging freeze'}


## 2. Authenticated Model-Selection Handoff

In [2]:
selection = load_and_validate_forecasting_model_selection_handoff(project_root=PROJECT.root, handoff_path=UPSTREAM, expected_dataset_slug='nottem')
print({'schema': selection['schema_version'], 'sha256': hashlib.sha256(UPSTREAM.read_bytes()).hexdigest(), 'selected_candidate': selection['selected_candidate_id'], 'readiness': selection['readiness']})

{'schema': 'forecasting-model-selection-handoff.v1', 'sha256': 'a7a24fa908a9500bc5e46c2f61b1cd1b0b03f3a441ed70a2b7f33d6e8c470398', 'selected_candidate': 'seasonal_trend_ols', 'readiness': {'baselines_evaluated': True, 'candidate_catalog_frozen_before_evaluation': True, 'candidate_models_evaluated': True, 'development_only_model_selection': True, 'final_holdout_evaluated': False, 'final_holdout_sealed': True, 'final_model_trained': False, 'final_model_training_ready': True, 'frozen_backtesting_contract_respected': True, 'metric_contract_frozen': True, 'model_artifact_materialized': False, 'model_bundle_materialized': False, 'model_selection_handoff_reloadable': True, 'operational_modeling_ready': False, 'preparation_handoff_validated': True, 'selected_specification_frozen': True, 'temporal_backtesting_completed': True}}


## 3. Frozen Selected Specification and Final-Training Contract

In [3]:
print(json.dumps(selection['selected_specification'], indent=2))
print(json.dumps(selection['final_training_instructions'], indent=2))

{
  "candidate_id": "seasonal_trend_ols",
  "complexity_rank": 2,
  "constructor": "statsmodels.regression.linear_model.OLS",
  "differencing_policy": "none",
  "exogenous_policy": "none_deterministic_calendar_is_not_source_exogenous",
  "family": "DeterministicSeasonalTrendOLS",
  "fixed_hyperparameters": {
    "calendar_month_dummies": 11,
    "intercept": true,
    "linear_time_trend": true,
    "reference_month": "January"
  },
  "learned_parameter_scope": "fit_from_scratch_on_each_training_fold_only",
  "multi_step_strategy": "direct_known_calendar_design_12_steps",
  "preprocessing_policy": "original_target_scale_no_global_learned_preprocessing",
  "role": "candidate",
  "seasonal_period": 12,
  "seasonal_policy": "11_calendar_month_dummies_january_reference",
  "trend_policy": "linear_time_trend"
}
{
  "do_not_change_candidate": true,
  "do_not_change_hyperparameters": true,
  "do_not_change_transform_policy": true,
  "do_not_retune": true,
  "evaluate_holdout_once": true,
  "fi

## 4. Full-Development Authentication
The authenticated development input is exactly 1920-01 through 1938-12 (228 rows).

## 5. Exact Selected-Model Reconstruction
The finalizer reconstructs intercept, linear time trend, and February–December indicators with January as reference.

## 6. Final Fit and Model Freeze
The first run fits once; equivalent reruns validate and reuse the existing complete set without fitting.

## 7. Trusted Staging Serialization Before Holdout Access

## 8. One-Time Final-Holdout Opening

## 9. Final 12-Month Forecast

In [4]:
result = run_forecasting_finalization(project_root=PROJECT.root)
print({'materialization_status': result.status, 'guard': dict(result.guard), 'artifact_paths': list(result.paths)})
handoff = load_and_validate_forecasting_final_model_handoff(project_root=PROJECT.root, handoff_path=OUTPUT/'final-model-handoff.json')
bundle = load_and_validate_forecasting_inference_bundle(project_root=PROJECT.root, bundle_path=OUTPUT/'inference-bundle.json')
manifest = load_and_validate_forecasting_final_model_manifest(project_root=PROJECT.root, manifest_path=OUTPUT/'final-model-manifest.json')
evidence = load_and_validate_forecasting_final_test_evidence(project_root=PROJECT.root, evidence_path=OUTPUT/'final-test-evidence.json')
model = load_trusted_forecasting_model_from_bundle(project_root=PROJECT.root, bundle_path=OUTPUT/'inference-bundle.json')
forecast_table = pd.DataFrame(evidence['forecasts'])[['forecast_period','horizon','y_pred']]
forecast_table

{'materialization_status': 'reused_equivalent', 'guard': {'final_fit_count': 0, 'holdout_open_count': 0, 'final_evaluation_count': 0}, 'artifact_paths': ['artifacts/models/nottem/final-pipeline.joblib', 'artifacts/models/nottem/final-model-manifest.json', 'artifacts/models/nottem/final-test-evidence.json', 'artifacts/models/nottem/inference-bundle.json', 'artifacts/models/nottem/final-model-handoff.json']}


,forecast_period,horizon,y_pred
0,1939-01,1,40.314766
1,1939-02,2,39.704240
2,1939-03,3,42.788450
3,1939-04,4,46.814766
4,1939-05,5,53.172661
5,1939-06,6,58.646345
6,1939-07,7,62.567398
7,1939-08,8,61.056871
8,1939-09,9,56.993713
9,1939-10,10,50.246345


## 10. Final MAE, RMSE, and Seasonal MASE

In [5]:
print({'metrics': evidence['metrics'], 'full_development_mase_denominator': evidence['final_seasonal_mase_denominator']})

{'metrics': {'mae': 1.526583820662746, 'rmse': 1.859966911452801, 'seasonal_mase_12': 0.5554954603489778}, 'full_development_mase_denominator': 2.7481481481481476}


## 11. Per-Horizon Final Forecast Evidence
Each horizon is one final-origin observation; these are per-horizon absolute errors, not averages across origins.

In [6]:
pd.DataFrame(evidence['forecasts'])[['forecast_period','horizon','y_true','y_pred','abs_error','scaled_abs_error']]

,forecast_period,horizon,y_true,y_pred,abs_error,scaled_abs_error
0,1939-01,1,39.4,40.314766,0.914766,0.332866
1,1939-02,2,40.9,39.704240,1.195760,0.435115
2,1939-03,3,42.4,42.788450,0.388450,0.141350
3,1939-04,4,47.8,46.814766,0.985234,0.358508
4,1939-05,5,52.4,53.172661,0.772661,0.281157
5,1939-06,6,58.0,58.646345,0.646345,0.235193
6,1939-07,7,60.7,62.567398,1.867398,0.679511
7,1939-08,8,61.8,61.056871,0.743129,0.270411
8,1939-09,9,58.2,56.993713,1.206287,0.438945
9,1939-10,10,46.7,50.246345,3.546345,1.290449


## 12. Backtesting-to-Final Generalization Review
This comparison is descriptive only and cannot trigger retuning or model replacement.

In [7]:
pd.DataFrame({'backtest': evidence['model_selection_reference_metrics'], 'final': evidence['metrics'], 'final_minus_backtest': evidence['generalization_deltas']})

,backtest,final,final_minus_backtest
mae,1.839438,1.526584,-0.312854
rmse,2.320801,1.859967,-0.460834
seasonal_mase_12,0.656960,0.555495,-0.101464


## 13. No-Retune and No-Model-Change Audit

In [8]:
print({k:evidence[k] for k in ['final_fit_count','final_forecast_call_count','final_evaluation_count','holdout_used_for_retuning','holdout_used_for_model_selection','candidate_changed_after_holdout','specification_changed_after_holdout']})

{'final_fit_count': 1, 'final_forecast_call_count': 1, 'final_evaluation_count': 1, 'holdout_used_for_retuning': False, 'holdout_used_for_model_selection': False, 'candidate_changed_after_holdout': False, 'specification_changed_after_holdout': False}


## 14. Forecasting Final Artifact Contract

In [9]:
artifact_audit = pd.DataFrame([{'name':p.name,'path':str(p.relative_to(PROJECT.root)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(OUTPUT.iterdir())])
artifact_audit

,name,path,sha256
0,final-model-handoff.json,artifacts/models/nottem/final-model-handoff.json,3a280491f5fa9e7d51db8e01c09dd202a2beddd3980f50...
1,final-model-manifest.json,artifacts/models/nottem/final-model-manifest.json,c6859e1425bbbf675ba169243e855ef153b6709c454341...
2,final-pipeline.joblib,artifacts/models/nottem/final-pipeline.joblib,3060d7a351688d7e60622d9112dbe01350e97964a01934...
3,final-test-evidence.json,artifacts/models/nottem/final-test-evidence.json,a46f34bab5c652f186f5b4055d7b1d018823a631ae8f9f...
4,inference-bundle.json,artifacts/models/nottem/inference-bundle.json,a74d8d67256550694f12131a295a018511ae0ef476c12b...


## 15. Atomic Final Artifact Materialization
A complete five-file set is staged, validated, and promoted as one directory. Partial sets fail closed.

## 16. Trusted Final Model and Bundle Reload

In [10]:
future = pd.period_range('1939-01', periods=12, freq='M', name='period')
replay = model.forecast_periods(future).to_numpy(float)
persisted = np.array([row['y_pred'] for row in evidence['forecasts']], dtype=float)
print({'model_artifact_kind': manifest['model_state']['model_artifact_kind'], 'model_state_fingerprint': model.model_state_semantic_fingerprint, 'trusted_replay_equal': bool(np.allclose(replay, persisted, rtol=1e-12, atol=1e-12))})

{'model_artifact_kind': 'frozen_forecasting_model', 'model_state_fingerprint': 'ddb57634c40d4f11192ca0478d40904a8d00e0e20f3aee679b46dc1117501279', 'trusted_replay_equal': True}


## 17. Forecasting Inference Input/Output Contract

In [11]:
print(json.dumps({'input_contract':bundle['input_contract'],'output_contract':bundle['output_contract']}, indent=2))

{
  "input_contract": {
    "exogenous_predictors": "none",
    "forecast_origin": "last supplied historical period",
    "historical_values_used_to_refit": false,
    "historical_values_used_to_update_coefficients": false,
    "history_required": true,
    "history_role": "establish forecast origin and validate monthly chronology",
    "kind": "monthly_univariate_history",
    "minimum_history_observations": 1,
    "period_requirements": {
      "contiguous": true,
      "monotonic_increasing": true,
      "monthly": true,
      "unique": true
    },
    "post_training_history_limitation": "values establish origin/context only; frozen coefficients are not updated",
    "refit_on_input": false,
    "representation": "period + temperature series/frame",
    "required_columns": [
      "period",
      "temperature"
    ],
    "supplied_history_end_minimum": "1938-12",
    "temperature_requirements": {
      "dtype": "numeric finite",
      "no_missing": true
    }
  },
  "output_contract

## 18. Notebook 05 Readiness
Notebook 05 is not implemented here. READY requires authenticated frozen artifacts and remains non-operational.

In [12]:
ready = handoff['readiness']['inference_demo_ready'] and not handoff['readiness']['operational_modeling_ready']
print({'Notebook 05 gate': 'READY' if ready else 'BLOCKED', 'readiness': handoff['readiness']})

{'Notebook 05 gate': 'READY', 'readiness': {'final_fit_count': 1, 'final_forecast_call_count': 1, 'final_holdout_evaluated': True, 'final_holdout_evaluation_count': 1, 'final_holdout_opened_after_freeze': True, 'final_model_handoff_reloadable': True, 'final_model_trained': True, 'final_test_evidence_materialized': True, 'final_training_completed': True, 'final_training_scope_validated': True, 'inference_bundle_materialized': True, 'inference_demo_ready': True, 'model_artifact_materialized': True, 'model_frozen': True, 'model_selection_handoff_validated': True, 'operational_modeling_ready': False, 'selected_specification_reconstructed': True}}
